# Notebook 01 — Tox21 Multi-task Benchmark
**Author: Himanshu Goel** | [Website](https://himanshugoel.github.io)

The Tox21 dataset (NIH/EPA/FDA) is the de-facto industry benchmark for computational toxicology. It contains ~8 000 compounds measured across **12 nuclear-receptor and stress-response endpoints**. This notebook covers the complete industry-standard pipeline: data loading, class-imbalance handling, DeepTox-style multi-task DNN, scaffold-split evaluation, and per-endpoint ROC-AUC.

| Panel | Endpoint | Biology |
|-------|----------|---------|
| Nuclear Receptor | NR-AR | Androgen receptor |
| Nuclear Receptor | NR-AhR | Aryl hydrocarbon |
| Nuclear Receptor | NR-ER | Estrogen receptor |
| Nuclear Receptor | NR-PPAR-gamma | PPAR-gamma |
| Stress Response | SR-ARE | Antioxidant response |
| Stress Response | SR-ATAD5 | Genotoxicity proxy |
| Stress Response | SR-MMP | Mitochondrial membrane |
| Stress Response | SR-p53 | DNA damage |

In [ ]:
!pip install deepchem rdkit scikit-learn pandas numpy matplotlib seaborn torch -q

In [ ]:
import deepchem as dc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score
import warnings; warnings.filterwarnings('ignore')

print("Loading Tox21 via DeepChem (industry standard loader)...")
tox21_tasks, datasets, transformers = dc.molnet.load_tox21(
    featurizer='ECFP', splitter='scaffold'   # scaffold split = realistic evaluation
)
train, valid, test = datasets
print(f"Tasks ({len(tox21_tasks)}): {tox21_tasks}")
print(f"Train: {len(train)} | Valid: {len(valid)} | Test: {len(test)}")
print(f"Feature dim: {train.X.shape[1]}")

## Class balance per endpoint

In [ ]:
y_tr, w_tr = train.y, train.w
stats = []
for i, task in enumerate(tox21_tasks):
    mask = w_tr[:, i] > 0
    vals = y_tr[:, i][mask]
    n_pos, n_neg = int(vals.sum()), int((vals==0).sum())
    stats.append({"Endpoint":task,"N_active":n_pos,"N_inactive":n_neg,
                  "Total":n_pos+n_neg,"Active_pct":round(n_pos/(n_pos+n_neg)*100,1)})
df = pd.DataFrame(stats).sort_values("Active_pct")
print(df.to_string(index=False))

fig,(ax1,ax2)=plt.subplots(1,2,figsize=(13,4))
colors=['#e74c3c' if r<8 else '#f39c12' if r<15 else '#27ae60' for r in df["Active_pct"]]
ax1.barh(df["Endpoint"],df["Active_pct"],color=colors)
ax1.axvline(15,color='k',linestyle='--',lw=0.8,label='15% line')
ax1.set_xlabel("% Active"); ax1.set_title("Class balance per Tox21 endpoint"); ax1.legend()
ax2.bar(df["Endpoint"],df["Total"],color='#2c3e50',alpha=0.7)
ax2.set_xticklabels(df["Endpoint"],rotation=45,ha='right')
ax2.set_ylabel("Compounds with label"); ax2.set_title("Data availability")
plt.tight_layout(); plt.savefig("tox21_eda.png",dpi=150); plt.show()

## DeepTox multi-task DNN (winning Tox21 Challenge architecture)

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader

class DeepToxNet(nn.Module):
    def __init__(self, in_f, n_tasks, hidden=[2048,1024,512,256], drop=0.35):
        super().__init__()
        layers=[]
        d=in_f
        for h in hidden:
            layers+=[nn.Linear(d,h),nn.BatchNorm1d(h),nn.ReLU(),nn.Dropout(drop)]
            d=h
        self.shared=nn.Sequential(*layers)
        self.heads=nn.ModuleList([nn.Linear(d,1) for _ in range(n_tasks)])
    def forward(self,x):
        h=self.shared(x)
        return torch.cat([head(h) for head in self.heads],dim=1)

X_tr=torch.FloatTensor(train.X); y_tt=torch.FloatTensor(train.y); w_tt=torch.FloatTensor(train.w)
X_va=torch.FloatTensor(valid.X)

model=DeepToxNet(X_tr.shape[1],len(tox21_tasks))
opt=torch.optim.Adam(model.parameters(),lr=1e-3,weight_decay=1e-5)
sched=torch.optim.lr_scheduler.StepLR(opt,step_size=5,gamma=0.5)

def masked_bce(pred,target,weight):
    mask=weight>0
    return F.binary_cross_entropy_with_logits(pred[mask],target[mask])

loader=DataLoader(TensorDataset(X_tr,y_tt,w_tt),batch_size=256,shuffle=True)
print(f"Model params: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
history={"loss":[],"auc":[]}
for epoch in range(20):
    model.train(); ep_loss=0
    for xb,yb,wb in loader:
        opt.zero_grad()
        loss=masked_bce(model(xb),yb,wb)
        loss.backward(); opt.step(); ep_loss+=loss.item()
    sched.step()
    model.eval()
    with torch.no_grad(): vp=torch.sigmoid(model(X_va)).numpy()
    aucs=[]
    for i in range(len(tox21_tasks)):
        mask=valid.w[:,i]>0
        if mask.sum()>10 and valid.y[:,i][mask].std()>0:
            try: aucs.append(roc_auc_score(valid.y[:,i][mask],vp[:,i][mask]))
            except: pass
    history["loss"].append(ep_loss/len(loader))
    history["auc"].append(np.mean(aucs) if aucs else 0)
    if epoch%5==0: print(f"Epoch {epoch:2d} | Loss={history['loss'][-1]:.4f} | Val AUC={history['auc'][-1]:.4f}")

In [ ]:
# Test set evaluation
X_te=torch.FloatTensor(test.X)
model.eval()
with torch.no_grad(): tp=torch.sigmoid(model(X_te)).numpy()
results=[]
for i,task in enumerate(tox21_tasks):
    mask=test.w[:,i]>0
    if mask.sum()>5:
        try: auc=roc_auc_score(test.y[:,i][mask],tp[:,i][mask])
        except: auc=0.5
    else: auc=None
    results.append({"Endpoint":task,"Test_AUC":round(auc,4) if auc else None,"N":int(mask.sum())})
res_df=pd.DataFrame(results).sort_values("Test_AUC",ascending=False)
print(res_df.to_string(index=False))
print(f"\nMean test AUC: {res_df.Test_AUC.mean():.4f}")

fig,ax=plt.subplots(figsize=(9,4))
colors=['#27ae60' if v>=0.8 else '#f39c12' if v>=0.7 else '#e74c3c' for v in res_df.Test_AUC.fillna(0.5)]
ax.barh(res_df.Endpoint,res_df.Test_AUC,color=colors)
ax.axvline(0.8,color='green',linestyle='--',lw=0.8,label='AUC 0.8')
ax.axvline(0.7,color='orange',linestyle='--',lw=0.8,label='AUC 0.7')
ax.set_xlim([0.5,1]); ax.set_title("DeepTox — Test AUC per endpoint"); ax.legend()
plt.tight_layout(); plt.savefig("tox21_auc.png",dpi=150); plt.show()

## Key takeaways
- Scaffold splitting is mandatory for realistic evaluation (random split inflates AUC ~10%)
- Missing labels (~30-50%) must be masked with weighted loss — never impute zeros
- SR-MMP (mitochondrial) is typically easiest; NR-AR-LBD the hardest endpoint
- Industry context: Tox21 winner DeepTox still competitive benchmark after a decade
- Regulatory: FDA ToxCast, EPA DSSTox, ICH S2/S7 guidelines reference this dataset